In [1]:
%pip install -U -q transformers datasets evaluate accelerate sentencepiece seqeval corus tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.4 MB/s eta 0:00:00


In [9]:
!curl -L -o lenta-ru-news.csv.gz 'https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  502M  100  502M    0     0  45.4M      0  0:00:11  0:00:11 --:--:-- 51.3M


In [10]:
from datasets import Dataset, DatasetDict, load_dataset
import numpy as np
import torch
from seqeval.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import (
    AutoConfig,
    AutoModelForMaskedLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorForTokenClassification,
    DataCollatorForWholeWordMask,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_callback import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback
import os
import pandas as pd
from corus import load_lenta
from tqdm.auto import tqdm
from datasets import concatenate_datasets

In [11]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "cointegrated/rubert-tiny2"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3

## Загрузка и анализ данных

In [15]:
ds_raw = load_dataset("gusevski/factrueval2016")
ds_raw

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train_data.json:   0%|          | 0.00/7.62M [00:00<?, ?B/s]

dev_data.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

test_data.json:   0%|          | 0.00/2.57M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

## Подготовка данных

In [16]:
def unwrap_split(split_ds):
    rows = split_ds[0]["data"]
    return Dataset.from_list(rows)

ds = DatasetDict({
    "train": unwrap_split(ds_raw["train"]),
    "validation": unwrap_split(ds_raw["validation"]),
    "test": unwrap_split(ds_raw["test"]),
})
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})

In [17]:
print(ds["train"][2]["tokens"])
print(ds["train"][2]["ner_tags"])
print(ds["train"][2]["ner_tags_str"])

['В', 'Ханты-Мансийском', 'автономном', 'округе', 'с', 'должности', 'снят', 'начальник', 'УВД', 'Николай', 'Гудожников', '.']
[0, 5, 6, 6, 0, 0, 0, 0, 3, 1, 2, 0]
['O', 'B-LOC', 'I-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'B-ORG', 'B-PER', 'I-PER', 'O']


Датасет `gusevski/factrueval2016`: после разворачивания сплиты train/val/test в пропорции 7746 / 2582 / 2582 предложений; в примере видны токены и строковые NER-метки (`ner_tags_str`).


In [18]:
all_tags = {tag for name in ds for ex in ds[name] for tag in ex["ner_tags_str"]}
label_names = sorted(all_tags)
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}
print(label_names)
print(label2id)
print(id2label)

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-PER', 3: 'I-LOC', 4: 'I-ORG', 5: 'I-PER', 6: 'O'}


Построены `label2id` / `id2label` по полному множеству строковых NER-меток на всех сплитах.


In [21]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_and_align_labels(example):
    t = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )
    word_ids = t.word_ids()
    ner = example["ner_tags_str"]
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[ner[word_idx]])
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    t["labels"] = label_ids
    return t

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [22]:
ds_tok = ds.map(
    tokenize_and_align_labels,
    remove_columns=ds["train"].column_names,
)
ds_tok

Map:   0%|          | 0/7746 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

Map:   0%|          | 0/2582 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})

In [ ]:
ex = ds_tok["train"][2]
print("input_ids", ex["input_ids"][:10])
print("tokens", tokenizer.convert_ids_to_tokens(ex["input_ids"][:10]))
print("labels", ex["labels"][:10])

input_ids [2, 282, 22585, 17, 20859, 24542, 62962, 17122, 329, 12238]
tokens ['[CLS]', 'В', 'Ханты', '-', 'Манси', '##йском', 'автономном', 'округе', 'с', 'должности']
labels [-100, 6, 0, -100, -100, -100, 3, 3, 6, 6]


Для `rubert-tiny2` с `is_split_into_words=True` метки переносятся на первый субтокен слова, остальным ставится `-100`; в примере видно согласованность `input_ids`, piece-токенов и `labels`.


In [29]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for pred_row, lab_row in zip(predictions, labels):
        pr_tags, lb_tags = [], []
        for p, lb in zip(pred_row, lab_row):
            if lb == -100:
                continue
            pr_tags.append(id2label[int(p)])
            lb_tags.append(id2label[int(lb)])
        true_preds.append(pr_tags)
        true_labels.append(lb_tags)
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
        "accuracy": accuracy_score(true_labels, true_preds),
    }

In [31]:
def swap_notebook_progress_callback(trainer):
    for cb in list(trainer.callback_handler.callbacks):
        if isinstance(cb, NotebookProgressCallback):
            trainer.remove_callback(cb)
            trainer.add_callback(ProgressCallback())
            break

In [35]:
def ner_metrics_from_evals(before_eval, after_eval):
    return {
        "before_f1": before_eval["eval_f1"],
        "after_f1": after_eval["eval_f1"],
        "before_precision": before_eval["eval_precision"],
        "after_precision": after_eval["eval_precision"],
        "before_recall": before_eval["eval_recall"],
        "after_recall": after_eval["eval_recall"],
        "before_accuracy": before_eval["eval_accuracy"],
        "after_accuracy": after_eval["eval_accuracy"],
    }

In [26]:
def ner_training_args(checkpoint_dir):
    return TrainingArguments(
        output_dir=checkpoint_dir,
        learning_rate=2e-5,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=torch.cuda.is_available(),
        report_to="none",
    )

## Baseline модель

In [ ]:
BASELINE_NER_DIR = "./hw3_ner_artifacts/baseline_ner"

config = AutoConfig.from_pretrained(BASE_MODEL)
config.num_labels = len(label2id)
config.id2label = id2label
config.label2id = label2id

model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, config=config)

training_args = ner_training_args(f"{BASELINE_NER_DIR}/trainer_checkpoints")
data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer)
trainer

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 14481.02it/s]
BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignore

In [ ]:
before_eval = trainer.evaluate()
trainer.train()
after_eval = trainer.evaluate()

ner_metrics = ner_metrics_from_evals(before_eval, after_eval)
ner_metrics

/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
100%|██████████| 162/162 [00:03<00:00, 43.69it/s]
                                                  
 33%|███▎      | 485/1455 [00:25<00:48, 20.19it/s]

{'eval_loss': '0.1341', 'eval_model_preparation_time': '0.0005', 'eval_precision': '0.7394', 'eval_recall': '0.821', 'eval_f1': '0.7781', 'eval_accuracy': '0.9643', 'eval_runtime': '1.773', 'eval_samples_per_second': '1456', 'eval_steps_per_second': '91.36', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 15.06it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 505/1455 [00:26<01:00, 15.61it/s]

{'loss': '0.3552', 'grad_norm': '0.9958', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


                                                  
 67%|██████▋   | 970/1455 [00:47<00:15, 31.46it/s]

...

{'eval_loss': '0.0818', 'eval_model_preparation_time': '0.0005', 'eval_precision': '0.8205', 'eval_recall': '0.8833', 'eval_f1': '0.8508', 'eval_accuracy': '0.9756', 'eval_runtime': '1.538', 'eval_samples_per_second': '1679', 'eval_steps_per_second': '105.3', 'epoch': '3'}


100%|██████████| 1455/1455 [01:08<00:00, 31.94it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '68.05', 'train_samples_per_second': '341.5', 'train_steps_per_second': '21.38', 'train_loss': '0.1878', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 88.52it/s] 


{'before_f1': 0.026586372253819556,
 'after_f1': 0.8507807447103389,
 'before_precision': 0.015041768669712036,
 'after_precision': 0.8205310996257351,
 'before_recall': 0.11435149654643131,
 'after_recall': 0.8833461243284727,
 'before_accuracy': 0.0692962919548314,
 'after_accuracy': 0.9755987965459305}

Итоговый F1 получился **0.851**, precision 0.821, recall 0.883, accuracy по токенам около 0.976. Сразу после случайной инициализации головы F1 был около 0.027, то есть за три эпохи удалось получить адекватное качество.

Такой baseline уже неплохо решает задачу, и его разумно принять за опору: все улучшения ниже будем измерять относительно F1 = **0.851** на этом же validation.


In [ ]:
trainer.save_model(BASELINE_NER_DIR)
tokenizer.save_pretrained(BASELINE_NER_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 19.65it/s]


('./hw3_ner_artifacts/baseline_ner/tokenizer_config.json',
 './hw3_ner_artifacts/baseline_ner/tokenizer.json')

## Дополнительное MLM

Перед NER мы добавили фазу masked language modeling.


In [ ]:
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)
mlm_model

Loading weights: 100%|██████████| 58/58 [00:00<00:00, 12111.40it/s]
BertForMaskedLM LOAD REPORT from: cointegrated/rubert-tiny2
Key                          | Status     |  | 
-----------------------------+------------+--+-
cls.seq_relationship.bias    | UNEXPECTED |  | 
cls.seq_relationship.weight  | UNEXPECTED |  | 
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_af

In [ ]:
mlm_text_rows_train = [{"text_for_mlm": " ".join(ex["tokens"])} for ex in ds["train"]]
mlm_text_rows_val = [{"text_for_mlm": " ".join(ex["tokens"])} for ex in ds["validation"]]
mlm_raw = Dataset.from_list(mlm_text_rows_train)
mlm_raw_val = Dataset.from_list(mlm_text_rows_val)
mlm_raw, mlm_raw_val

(Dataset({
     features: ['text_for_mlm'],
     num_rows: 7746
 }),
 Dataset({
     features: ['text_for_mlm'],
     num_rows: 2582
 }))

In [ ]:
def tokenize_for_mlm(examples):
    return tokenizer(
        examples["text_for_mlm"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_special_tokens_mask=True,
        return_offsets_mapping=True,
    )


mlm_ds = mlm_raw.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=mlm_raw.column_names,
)
mlm_ds_val = mlm_raw_val.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=mlm_raw_val.column_names,
)

mlm_ds, mlm_ds_val

Map: 100%|██████████| 2582/2582 [00:00<00:00, 6081.85 examples/s]


(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
     num_rows: 7746
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask', 'offset_mapping'],
     num_rows: 2582
 }))

In [ ]:
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)
mlm_collator

DataCollatorForLanguageModeling(tokenizer=BertTokenizer(name_or_path='cointegrated/rubert-tiny2', vocab_size=83828, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=False, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_te

In [ ]:
def mlm_training_args(checkpoint_dir):
    return TrainingArguments(
        output_dir=checkpoint_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=5e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        fp16=torch.cuda.is_available(),
        report_to="none",
        remove_unused_columns=False,
    )

In [ ]:
MLM_RANDOM_MASKING_DIR = "./hw3_ner_artifacts/mlm_random_masking"

mlm_training_args_config = mlm_training_args(f"{MLM_RANDOM_MASKING_DIR}/trainer_checkpoints")
mlm_training_args_config

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [ ]:
mlm_trainer = Trainer(
    model=mlm_model,
    args=mlm_training_args_config,
    train_dataset=mlm_ds,
    eval_dataset=mlm_ds_val,
    data_collator=mlm_collator,
    processing_class=tokenizer,
)

swap_notebook_progress_callback(mlm_trainer)

metrics_mlm_before = mlm_trainer.evaluate()
mlm_trainer.train()
metrics_mlm_after = mlm_trainer.evaluate()

 33%|███▎      | 485/1455 [03:04<06:06,  2.65it/s]

{'eval_loss': '3.263', 'eval_model_preparation_time': '0.0003', 'eval_runtime': '16.16', 'eval_samples_per_second': '159.8', 'eval_steps_per_second': '10.02', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.91it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 34%|███▍      | 500/1455 [03:10<06:06,  2.61it/s]  

{'loss': '3.35', 'grad_norm': '14.67', 'learning_rate': '3.285e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [06:08<02:28,  3.27it/s]

...

100%|██████████| 1455/1455 [09:08<00:00,  3.19it/s]

{'eval_loss': '3.188', 'eval_model_preparation_time': '0.0003', 'eval_runtime': '16.11', 'eval_samples_per_second': '160.3', 'eval_steps_per_second': '10.06', 'epoch': '3'}


100%|██████████| 1455/1455 [09:08<00:00,  3.19it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].
There were unexpected keys in the checkpo

{'train_runtime': '548.9', 'train_samples_per_second': '42.34', 'train_steps_per_second': '2.651', 'train_loss': '3.177', 'epoch': '3'}


100%|██████████| 162/162 [00:16<00:00,  9.96it/s]


In [ ]:
mlm_run_metrics = {
    "mlm_loss_before": metrics_mlm_before["eval_loss"],
    "mlm_loss_after": metrics_mlm_after["eval_loss"],
}
mlm_run_metrics

{'mlm_loss_before': 3.5802831649780273, 'mlm_loss_after': 3.170499563217163}

In [ ]:
mlm_trainer.save_model(MLM_RANDOM_MASKING_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.52it/s]


Здесь использовали обычное случайное маскирование на уровне субтокенов и три эпохи MLM на train. По валидационному loss задача явно учится: значение упало с 3.58 до 3.17.


## Дообучение NER после MLM

In [ ]:
ner_config_mlm = AutoConfig.from_pretrained(MLM_RANDOM_MASKING_DIR)
ner_config_mlm.num_labels = len(label2id)
ner_config_mlm.id2label = id2label
ner_config_mlm.label2id = label2id

model_mlm_ner = AutoModelForTokenClassification.from_pretrained(
    MLM_RANDOM_MASKING_DIR,
    config=ner_config_mlm,
)
model_mlm_ner

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 12247.83it/s]
BertForTokenClassification LOAD REPORT from: ./hw3_ner_artifacts/mlm_random_masking
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, ele

In [ ]:
NER_MLM_DIR = "./hw3_ner_artifacts/mlm_ner"

ner_mlm_training_args = ner_training_args(f"{NER_MLM_DIR}/trainer_checkpoints")
ner_mlm_training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [ ]:
ner_mlm_collator = DataCollatorForTokenClassification(tokenizer)

trainer_mlm_ner = Trainer(
    model=model_mlm_ner,
    args=ner_mlm_training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=ner_mlm_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer_mlm_ner)
trainer_mlm_ner

In [ ]:
before_eval_mlm = trainer_mlm_ner.evaluate()
trainer_mlm_ner.train()
after_eval_mlm = trainer_mlm_ner.evaluate()

ner_metrics_mlm = ner_metrics_from_evals(before_eval_mlm, after_eval_mlm)
ner_metrics_mlm

 33%|███▎      | 485/1455 [00:17<00:31, 31.08it/s]

{'eval_loss': '0.1144', 'eval_model_preparation_time': '0.0009', 'eval_precision': '0.7886', 'eval_recall': '0.8425', 'eval_f1': '0.8147', 'eval_accuracy': '0.9692', 'eval_runtime': '1.516', 'eval_samples_per_second': '1703', 'eval_steps_per_second': '106.9', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.29it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 505/1455 [00:18<01:02, 15.30it/s]

{'loss': '0.3301', 'grad_norm': '1.169', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [00:33<00:12, 38.17it/s]

...

{'eval_loss': '0.07459', 'eval_model_preparation_time': '0.0009', 'eval_precision': '0.8531', 'eval_recall': '0.8914', 'eval_f1': '0.8718', 'eval_accuracy': '0.978', 'eval_runtime': '1.386', 'eval_samples_per_second': '1863', 'eval_steps_per_second': '116.9', 'epoch': '3'}


100%|██████████| 1455/1455 [00:49<00:00, 35.48it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '49.01', 'train_samples_per_second': '474.1', 'train_steps_per_second': '29.69', 'train_loss': '0.1709', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 113.06it/s]


{'before_f1': 0.037565905096660815,
 'after_f1': 0.8718333646087447,
 'before_precision': 0.021211670139922596,
 'after_precision': 0.8531031950055087,
 'before_recall': 0.1640445126630852,
 'after_recall': 0.8914044512663085,
 'before_accuracy': 0.09844488727386395,
 'after_accuracy': 0.9779627241823936}

In [ ]:
{
    "baseline_after_f1": ner_metrics["after_f1"],
    "mlm_pretrained_after_f1": ner_metrics_mlm["after_f1"],
    "delta_f1": ner_metrics_mlm["after_f1"] - ner_metrics["after_f1"],
}

{'baseline_after_f1': 0.8507807447103389,
 'mlm_pretrained_after_f1': 0.8718333646087447,
 'delta_f1': 0.02105261989840579}

Мы взяли чекпоинт после random MLM, навесили свежую NER-голову и обучали три эпохи на размеченном train. На том же val получили F1 около **0.872** , precision 0.853, recall 0.891, accuracy по токенам 0.978. 

По мтерике F1 это примерно на **0.021** F1 выше чем показа baseline модель. 

Таким образом, дополнительное языковое предобучение реально помогло, но прорывного результата не дало.


## MLM с Whole Word Masking

Дальше мы повторили MLM на тех же текстах train, но маскировали целые слова (whole word masking), а не отдельные куски. Затем снова планировалось обучить NER, чтобы сравнить с вариантом со случайным маскированием.


In [ ]:
mlm_whole_word_collator = DataCollatorForWholeWordMask(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)
mlm_whole_word_collator

/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/transformers/data/data_collator.py:1028: FutureWarning: DataCollatorForWholeWordMask is deprecated and will be removed in a future version, you can now use DataCollatorForLanguageModeling with whole_word_mask=True instead.
  warnings.warn(


DataCollatorForWholeWordMask(tokenizer=BertTokenizer(name_or_path='cointegrated/rubert-tiny2', vocab_size=83828, model_max_length=2048, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}), mlm=True, whole_word_mask=True, mlm_probability=0.15, mask_replace_prob=0.8, random_replace_prob=0.1, pad_to_multiple_of=None, return_tensor

In [ ]:
MLM_WWM_DIR = "./hw3_ner_artifacts/mlm_whole_word"

mlm_wwm_training_args = mlm_training_args(
    f"{MLM_WWM_DIR}/trainer_checkpoints",
)

In [ ]:
mlm_wwm_trainer = Trainer(
    model=mlm_model,
    args=mlm_wwm_training_args,
    train_dataset=mlm_ds,
    eval_dataset=mlm_ds_val,
    data_collator=mlm_whole_word_collator,
    processing_class=tokenizer,
)

swap_notebook_progress_callback(mlm_wwm_trainer)

metrics_mlm_wwm_before = mlm_wwm_trainer.evaluate()
mlm_wwm_trainer.train()
metrics_mlm_wwm_after = mlm_wwm_trainer.evaluate()

 33%|███▎      | 485/1455 [03:10<05:07,  3.15it/s]

{'eval_loss': '4.051', 'eval_model_preparation_time': '0.0004', 'eval_runtime': '20.05', 'eval_samples_per_second': '128.8', 'eval_steps_per_second': '8.08', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.03it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 34%|███▍      | 500/1455 [03:18<07:13,  2.21it/s]  

{'loss': '3.494', 'grad_norm': '15.4', 'learning_rate': '3.285e-05', 'epoch': '1.031'}


 67%|██████▋   | 970/1455 [06:23<02:44,  2.95it/s]

...

100%|██████████| 1455/1455 [09:22<00:00,  2.94it/s]

{'eval_loss': '4.067', 'eval_model_preparation_time': '0.0004', 'eval_runtime': '16.78', 'eval_samples_per_second': '153.9', 'eval_steps_per_second': '9.654', 'epoch': '3'}


100%|██████████| 1455/1455 [09:22<00:00,  2.94it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.decoder.bias'].
There were unexpected keys in the checkpo

{'train_runtime': '563', 'train_samples_per_second': '41.28', 'train_steps_per_second': '2.584', 'train_loss': '3.576', 'epoch': '3'}


100%|██████████| 162/162 [00:16<00:00,  9.88it/s]


In [ ]:
mlm_wwm_metrics = {
    "mlm_loss_before": metrics_mlm_wwm_before["eval_loss"],
    "mlm_loss_after": metrics_mlm_wwm_after["eval_loss"],
}
mlm_wwm_metrics

{'mlm_loss_before': 4.273593902587891, 'mlm_loss_after': 4.0600504875183105}

In [ ]:
mlm_wwm_trainer.save_model(MLM_WWM_DIR)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.23it/s]


In [ ]:
ner_config_mlm_wwm = AutoConfig.from_pretrained(MLM_WWM_DIR)
ner_config_mlm_wwm.num_labels = len(label2id)
ner_config_mlm_wwm.id2label = id2label
ner_config_mlm_wwm.label2id = label2id

In [ ]:
model_mlm_wwm_ner = AutoModelForTokenClassification.from_pretrained(
    MLM_WWM_DIR,
    config=ner_config_mlm_wwm,
)
model_mlm_wwm_ner

Loading weights: 100%|██████████| 53/53 [00:00<00:00, 14114.17it/s]
BertForTokenClassification LOAD REPORT from: ./hw3_ner_artifacts/mlm_whole_word
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, ele

In [ ]:
NER_MLM_WWM_DIR = "./hw3_ner_artifacts/mlm_wwm_ner"

ner_mlm_wwm_training_args = ner_training_args(f"{NER_MLM_WWM_DIR}/trainer_checkpoints")
ner_mlm_wwm_training_args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [ ]:
ner_mlm_wwm_collator = DataCollatorForTokenClassification(tokenizer)

trainer_mlm_wwm_ner = Trainer(
    model=model_mlm_wwm_ner,
    args=ner_mlm_wwm_training_args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=ner_mlm_wwm_collator,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer_mlm_wwm_ner)
trainer_mlm_wwm_ner

In [ ]:
before_eval_mlm_wwm = trainer_mlm_wwm_ner.evaluate()
trainer_mlm_wwm_ner.train()
after_eval_mlm_wwm = trainer_mlm_wwm_ner.evaluate()

  0%|          | 0/162 [00:00<?, ?it/s]

 33%|███▎      | 485/1455 [00:15<00:28, 33.78it/s]

{'eval_loss': '0.1195', 'eval_model_preparation_time': '0.0008', 'eval_precision': '0.7843', 'eval_recall': '0.8273', 'eval_f1': '0.8052', 'eval_accuracy': '0.9674', 'eval_runtime': '1.361', 'eval_samples_per_second': '1898', 'eval_steps_per_second': '119.1', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.85it/s]
/Users/user/my_projects/dl_nlp_2026/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
 35%|███▍      | 507/1455 [00:16<00:48, 19.71it/s]

{'loss': '0.3319', 'grad_norm': '1.099', 'learning_rate': '1.314e-05', 'epoch': '1.031'}


...

100%|██████████| 1455/1455 [00:47<00:00, 33.66it/s]

{'eval_loss': '0.07859', 'eval_model_preparation_time': '0.0008', 'eval_precision': '0.8474', 'eval_recall': '0.883', 'eval_f1': '0.8648', 'eval_accuracy': '0.9766', 'eval_runtime': '1.407', 'eval_samples_per_second': '1835', 'eval_steps_per_second': '115.1', 'epoch': '3'}


100%|██████████| 1455/1455 [00:47<00:00, 33.66it/s]There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encode

{'train_runtime': '47.79', 'train_samples_per_second': '486.3', 'train_steps_per_second': '30.45', 'train_loss': '0.1723', 'epoch': '3'}


100%|██████████| 162/162 [00:01<00:00, 94.00it/s] 


In [ ]:
ner_metrics_mlm_wwm = ner_metrics_from_evals(before_eval_mlm_wwm, after_eval_mlm_wwm)
ner_metrics_mlm_wwm

{'before_f1': 0.03740929373056295,
 'after_f1': 0.8647937611575683,
 'before_precision': 0.02119401460599561,
 'after_precision': 0.8473577610016572,
 'before_recall': 0.15924788948580199,
 'after_recall': 0.8829623944742901,
 'before_accuracy': 0.11749306450982691,
 'after_accuracy': 0.9765951627398116}

In [ ]:
{
    "baseline_after_f1": ner_metrics["after_f1"],
    "mlm_pretrained_after_f1": ner_metrics_mlm_wwm["after_f1"],
    "delta_f1": ner_metrics_mlm_wwm["after_f1"] - ner_metrics["after_f1"],
}

{'baseline_after_f1': 0.8507807447103389,
 'mlm_pretrained_after_f1': 0.8647937611575683,
 'delta_f1': 0.01401301644722941}

После WWM-MLM мы снова дообучили модель на NER в течение трёх эпох.

В итоге получили F1 примерно равный **0.865**, precision 0.847 и recall 0.883. Это лучше baseline (0.851), прирост около **+0.014** по F1. Однако результат всё равно хуже, чем у варианта с обычным MLM (0.872).

Получается, что маскирование целых слов действительно даёт выигрыш по сравнению с чистым fine-tuning, но в этом эксперименте уступает стандартному случайному маскированию.

Если сравнить все варианты на текущий момент, то baseline по F1 дает **0.851**, random MLM + NER - **0.87**2**, а WWM + NER - **0.865**. Лидером остаётся модель с обычным MLM-предобучением.

## Синтетическая разметка (teacher)

Дальше мы пробовали увеличить train за счёт автоматической разметки новостей Lenta и смешали эти псевдо-примеры с исходным gold.


In [4]:
from transformers import pipeline

TEACHER_MODEL = "Gherman/bert-base-NER-Russian"
_dev = 0 if torch.cuda.is_available() else -1
ner_pipeline = pipeline(
    "ner",
    model=TEACHER_MODEL,
    tokenizer=TEACHER_MODEL,
    device=_dev,
    batch_size=16,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/709M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [5]:
def normalize_teacher_tag(raw_tag: str) -> str | None:
    tag = raw_tag.upper()
    if any(x in tag for x in ("FIRST_NAME", "LAST_NAME", "MIDDLE_NAME", "PERSON")):
        return "PER"
    if any(
        x in tag
        for x in (
            "CITY",
            "COUNTRY",
            "REGION",
            "DISTRICT",
            "STREET",
            "HOUSE",
            "LOCATION",
        )
    ):
        return "LOC"
    if any(x in tag for x in ("ORG", "ORGANIZATION", "COMPANY")):
        return "ORG"
    return None

In [6]:
def _word_char_spans(tokens: list[str]) -> list[tuple[int, int]]:
    spans: list[tuple[int, int]] = []
    pos = 0
    for tok in tokens:
        spans.append((pos, pos + len(tok)))
        pos += len(tok) + 1
    return spans


def teacher_preds_to_bio(tokens: list[str], preds: list[dict]) -> list[str]:
    labels = ["O"] * len(tokens)
    spans = _word_char_spans(tokens)
    used: set[int] = set()

    for pred in preds:
        ent_type = normalize_teacher_tag(pred["entity"])
        if ent_type is None:
            continue
        start_char, end_char = pred["start"], pred["end"]
        covered: list[int] = []
        for i, (s, e) in enumerate(spans):
            if not (e <= start_char or s >= end_char):
                covered.append(i)
        if not covered:
            continue
        covered = [i for i in covered if i not in used]
        if not covered:
            continue
        labels[covered[0]] = f"B-{ent_type}"
        for j in covered[1:]:
            labels[j] = f"I-{ent_type}"
        used.update(covered)

    return labels

In [7]:
def fix_bio_for_project(tags: list[str]) -> list[str]:
    out: list[str] = []
    for t in tags:
        if t.startswith("I-"):
            et = t.split("-", 1)[1]
            prev = out[-1] if out else "O"
            if prev not in (f"B-{et}", f"I-{et}"):
                t = f"B-{et}"
        out.append(t)
    merged: list[str] = []
    for t in out:
        if t.startswith("B-"):
            et = t.split("-", 1)[1]
            if merged and merged[-1] == f"B-{et}":
                t = f"I-{et}"
        merged.append(t)
    return merged

### Lenta: выборка

Новости читали из `lenta-ru-news.csv.gz` потоково и в выборку попадали только строки с непустым полем topic.

Это отдельный новостной домен, он заметно отличается от стиля factRuEval, и это важно помнить при интерпретации псевдо-разметки.


In [12]:
LENTA_PATH = "lenta-ru-news.csv.gz"
N_SAMPLES = 10_000

if not os.path.isfile(LENTA_PATH):
    raise FileNotFoundError(
        f"Нет файла {os.path.abspath(LENTA_PATH)}. Скачайте lenta-ru-news.csv.gz или поправьте путь."
    )

rows: list[dict] = []
try:
    it = load_lenta(LENTA_PATH)
    with tqdm(desc="Loading lenta", total=N_SAMPLES) as pbar:
        for rec in it:
            topic = str(rec.topic or "").strip()
            if not topic:
                continue
            rows.append({"title": rec.title, "text": rec.text, "topic": topic})
            pbar.update(1)
            if len(rows) >= N_SAMPLES:
                break
except EOFError:
    pass

lenta_df = pd.DataFrame(rows)
lenta_df["topic"] = lenta_df["topic"].astype(str).str.strip()
lenta_df["title"] = lenta_df["title"].fillna("").astype(str).str.strip()
lenta_df["text"] = lenta_df["text"].fillna("").astype(str).str.strip()

df_sample = lenta_df.reset_index(drop=True)

len(df_sample)


Loading lenta:   0%|          | 0/10000 [00:00<?, ?it/s]

10000

In [13]:
def whitespace_tokenize(text: str, max_words: int = 256) -> list[str]:
    return text.split()[:max_words]


df_sample["full_text"] = (df_sample["title"] + ". " + df_sample["text"]).str.strip()
df_sample["full_text"] = df_sample["full_text"].str.replace(r"\s+", " ", regex=True)

df_sample["tokens"] = df_sample["full_text"].apply(
    lambda t: whitespace_tokenize(t, max_words=256),
)
df_sample = df_sample[df_sample["tokens"].map(len) > 0].reset_index(drop=True)

len(df_sample)


10000

In [19]:
synth_rows: list[dict] = []
n = len(df_sample)

for start in tqdm(range(0, n, BATCH_SIZE), desc="Pseudo-labeling"):
    end = min(start + BATCH_SIZE, n)
    batch_tokens = [df_sample.iloc[i]["tokens"] for i in range(start, end)]
    batch_texts = [" ".join(toks) for toks in batch_tokens]
    batch_preds = ner_pipeline(batch_texts)
    for i, tokens, preds in zip(range(start, end), batch_tokens, batch_preds):
        bio_tags = fix_bio_for_project(teacher_preds_to_bio(tokens, preds))
        if len(tokens) != len(bio_tags):
            continue
        synth_rows.append(
            {
                "id": f"synth-{i}",
                "tokens": tokens,
                "length": len(tokens),
                "ner_tags_str": bio_tags,
                "ner_tags": [label2id[t] for t in bio_tags],
            }
        )

len(synth_rows)


Pseudo-labeling:   0%|          | 0/625 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


10000

In [23]:
ds_synth = Dataset.from_list(synth_rows)
ds_tok_synth = ds_synth.map(
    tokenize_and_align_labels,
    remove_columns=ds_synth.column_names,
)
ds_synth, ds_tok_synth

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

(Dataset({
     features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
     num_rows: 10000
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
     num_rows: 10000
 }))

In [24]:
ds_tok_train_combined = concatenate_datasets([ds_tok["train"], ds_tok_synth]).shuffle(seed=42)
ds_tok_train_combined

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 17746
})

In [27]:
COMBINED_NER_DIR = "./hw3_ner_artifacts/ner_combined_teacher"

config_combined = AutoConfig.from_pretrained(BASE_MODEL)
config_combined.num_labels = len(label2id)
config_combined.id2label = id2label
config_combined.label2id = label2id

model_combined = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, config=config_combined)
training_args_combined = ner_training_args(f"{COMBINED_NER_DIR}/trainer_checkpoints")
collator_combined = DataCollatorForTokenClassification(tokenizer)

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expec

In [32]:
trainer_combined = Trainer(
    model=model_combined,
    args=training_args_combined,
    train_dataset=ds_tok_train_combined,
    eval_dataset=ds_tok["validation"],
    processing_class=tokenizer,
    data_collator=collator_combined,
    compute_metrics=compute_metrics,
)
swap_notebook_progress_callback(trainer_combined)
trainer_combined

In [33]:
before_eval_combined = trainer_combined.evaluate()
trainer_combined.train()
after_eval_combined = trainer_combined.evaluate()

  0%|          | 0/162 [00:00<?, ?it/s]

  0%|          | 0/3330 [00:00<?, ?it/s]

{'loss': '0.2317', 'grad_norm': '0.3402', 'learning_rate': '1.7e-05', 'epoch': '0.4505'}
{'loss': '0.06537', 'grad_norm': '0.7315', 'learning_rate': '1.4e-05', 'epoch': '0.9009'}


  0%|          | 0/162 [00:00<?, ?it/s]

{'eval_loss': '0.1324', 'eval_model_preparation_time': '0.0019', 'eval_precision': '0.752', 'eval_recall': '0.8127', 'eval_f1': '0.7812', 'eval_accuracy': '0.9617', 'eval_runtime': '1.804', 'eval_samples_per_second': '1431', 'eval_steps_per_second': '89.8', 'epoch': '1'}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

{'train_runtime': '186.1', 'train_samples_per_second': '286', 'train_steps_per_second': '17.89', 'train_loss': '0.07425', 'epoch': '3'}


  0%|          | 0/162 [00:00<?, ?it/s]

NameError: name 'ner_metrics_from_evals' is not defined

In [36]:
ner_metrics_combined = ner_metrics_from_evals(before_eval_combined, after_eval_combined)
ner_metrics_combined

{'before_f1': 0.027943207807794204,
 'after_f1': 0.8457174741932484,
 'before_precision': 0.01584122827286965,
 'after_precision': 0.8206099981952716,
 'before_recall': 0.11838066001534919,
 'after_recall': 0.8724098234842671,
 'before_accuracy': 0.054272652678466766,
 'after_accuracy': 0.972629234556324}

In [37]:
trainer_combined.save_model(COMBINED_NER_DIR)
tokenizer.save_pretrained(COMBINED_NER_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./hw3_ner_artifacts/ner_combined_teacher/tokenizer_config.json',
 './hw3_ner_artifacts/ner_combined_teacher/tokenizer.json')

In [38]:
ner_compare_teacher_synth = {
    "baseline_after_f1": "0.8507807447103389", # ner_metrics["after_f1"],
    "mlm_ner_after_f1": "0.8718333646087447", # ner_metrics_mlm["after_f1"],
    "mlm_wwm_ner_after_f1": "0.8647937611575683", # ner_metrics_mlm_wwm["after_f1"],
    "combined_teacher_after_f1": ner_metrics_combined["after_f1"],
}
ner_compare_teacher_synth

{'baseline_after_f1': '0.8507807447103389',
 'mlm_ner_after_f1': '0.8718333646087447',
 'mlm_wwm_ner_after_f1': '0.8647937611575683',
 'combined_teacher_after_f1': 0.8457174741932484}

### Синтетическая разметка: выводы

В качестве teacher использовали модель `Gherman/bert-base-NER-Russian`.  
Для данных взяли статьи из Lenta: объединяли заголовок и текст, обрезали примерно до 256 слов и прогоняли через модель.  
Далее предсказания приводили к типам PER, LOC и ORG, строили BIO-разметку и добавляли эти данные к исходному train.

На валидации получили **F1 = 0.846**, **precision = 0.821**, **recall = 0.872**.  

Это хуже baseline (**0.851**) и хуже лучшего варианта с MLM (**0.872**).

Скорее всего, проблема в доменном сдвиге: тексты Lenta отличаются от factRuEval, а teacher допускает ошибки в границах и типах сущностей.  
Дополнительно влияет обрезка длинных текстов, из-за чего теряется контекст.  
В сумме шум в псевдоразметке перевешивает пользу от увеличения объёма данных.

Итог: в текущем виде добавление синтетической разметки не дало улучшения и даже немного ухудшило качество модели.

## Итог

В задаче NER на factRuEval2016 использовали модель `cointegrated/rubert-tiny2`.  
Все эксперименты сравнивались на одном и том же validation через seqeval F1 после трёх эпох обучения.

Baseline дал **F1 = 0.851**.

Добавление этапа MLM с обычным случайным маскированием перед NER дало лучший результат — **F1 = 0.872**, что примерно на +0.02 выше baseline.  
Модель успевает лучше адаптироваться к тексту до обучения на разметке, и это даёт стабильный прирост.

Whole Word Masking тоже улучшает качество: **F1 = 0.865**.  
Это выше baseline, но заметно хуже, чем при обычном MLM, поэтому в данном эксперименте random masking оказался эффективнее.

Синтетическая разметка показала худший результат — **F1 = 0.846**, что ниже baseline.  
Добавление псевдо-данных из другого домена привело к шуму, и модель начала хуже обобщать.

Итоговое сравнение:
- baseline — **0.851**
- random MLM + NER — **0.872**
- WWM + NER — **0.865**
- gold + pseudo — **0.846**

Лучший результат даёт связка с обычным MLM перед NER.  
Дополнительные усложнения вроде WWM или псевдоразметки в этом запуске не дали преимущества.

Если развивать дальше:
- попробовать более качественную teacher-модель
- фильтровать псевдоразметку по уверенности
- использовать данные ближе к целевому домену
- отдельно подбирать параметры для MLM-этапа